In [ ]:
import os 
import pandas as pd
import os
from ase.io import read

# 병합할 xyz 파일을 찾을 폴더 리스트
folders = [
    "xtbopt/02_python_ase_scf2",
    "xtbopt/04_python_ase_scf_after_03_xtb_opt"
]

xyz_files = []

# 폴더에서 모든 .xyz 파일 찾기
for folder in folders:
    for f in os.listdir(folder):
        if f.endswith(".xyz"):
            xyz_files.append(os.path.join(folder, f))

print(f"✅ Found {len(xyz_files)} XYZ files:")
for f in xyz_files:
    print("  └", f)

# 모든 xyz 파일을 읽어서 하나의 db 리스트로 병합
db = []
for xyz in xyz_files:
    frames = read(xyz, ":")  # ':' = all frames
    db.extend(frames)
    print(len(frames))
print(f"\n✅ Total merged frames = {len(db)}")
from ase.io import write
write("00_makedataset_merged_db.xyz", db)

✅ Found 6 XYZ files:
  └ xtbopt/02_python_ase_scf2\ionpairs_xtb.xyz
  └ xtbopt/04_python_ase_scf_after_03_xtb_opt\ionpairs_xtb_sample_1.xyz
  └ xtbopt/04_python_ase_scf_after_03_xtb_opt\ionpairs_xtb_sample_2.xyz
  └ xtbopt/04_python_ase_scf_after_03_xtb_opt\ionpairs_xtb_sample_3.xyz
  └ xtbopt/04_python_ase_scf_after_03_xtb_opt\ionpairs_xtb_sample_4.xyz
  └ xtbopt/04_python_ase_scf_after_03_xtb_opt\ionpairs_xtb_sample_5.xyz
100
143
143
143
142
142

✅ Total merged frames = 813


In [15]:
from ase import Atoms
from ase.io import read, write
from xtb.ase.calculator import XTB
from tqdm import tqdm
from aseMolec import extAtoms as ea
from collections import Counter
# 1. Load merged db from xyz
db = read("./00_makedataset_merged_db.xyz", ":")

# 2. Add isolated atoms for EMIM + TFSI system
isolated_atoms = ["H", "C", "N", "O", "F", "S"]
iso_db = [Atoms(sym) for sym in isolated_atoms]

# Tag isolated atom configs
for at in iso_db:
    at.info["config_type"] = "isolated_atom"

# 3. Run xTB single-point for isolated atoms
xtb_calc = XTB(method="GFN2-xTB")
for at in tqdm(iso_db, desc="Running xTB for isolated atoms"):
    at.calc = xtb_calc
    at.info["energy_xtb"] = at.get_potential_energy()
    at.arrays["forces_xtb"] = at.get_forces()

# 4. Combine isolated atoms + dataset
db = iso_db + db
print("✅ Final DB size =", len(db))

# 5. Save updated dataset
write("00_makedataset_with_iso.extxyz", db)
print("✅ Saved as 00_makedataset_with_iso.extxyz")


Running xTB for isolated atoms: 100%|██████████| 6/6 [00:00<00:00, 99.12it/s]


✅ Final DB size = 819
✅ Saved as 00_makedataset_with_iso.extxyz


In [16]:
db = read('./00_makedataset_with_iso.xyz', ':15')

print("E0s: \n", ea.get_E0(db, tag='_xtb'))
print("Total energy per config: \n", ea.get_prop(db, 'info', 'energy_xtb', peratom=False)[13])
print("Toal energy per atom: \n", ea.get_prop(db, 'info', 'energy_xtb', peratom=True)[13])
print("Atomization energy per config: \n", ea.get_prop(db, 'bind', prop='_xtb', peratom=False)[13])
print("Atomization energy per atom: \n", ea.get_prop(db, 'bind', prop='_xtb', peratom=True)[13])

E0s: 
 {'H': np.float64(-10.707211383396714), 'C': np.float64(-48.847445262804705), 'N': np.float64(-71.00681805517411), 'O': np.float64(-102.57117256025786), 'F': np.float64(-125.69864294466228), 'S': np.float64(-85.66881795502964)}
Total energy per config: 
 -6046.882122868263
Toal energy per atom: 
 -52.12829416265744
Atomization energy per config: 
 -655.344248268284
Atomization energy per atom: 
 -5.649519381623138


In [18]:
import numpy as np
from ase.io import write

# ✅ 전체 db 다시 읽기 (이미 있다면 재사용)
db = read('./00_makedataset_with_iso.xyz', ':')

# ✅ 섞기 (random shuffle)
np.random.seed(42)  # 재현 가능성 유지
indices = np.random.permutation(len(db))
db = [db[i] for i in indices]

# ✅ train/test split 비율 정의
train_ratio = 0.5
n_train = int(len(db) * train_ratio)

train_db = db[:n_train]
test_db  = db[n_train:]

print(f"✅ Total frames: {len(db)}")
print(f"✅ Train frames: {len(train_db)}")
print(f"✅ Test frames:  {len(test_db)}")

# ✅ 저장 (extxyz 추천)
write(f"dataset_train_{train_ratio}.extxyz", train_db)
write(f"dataset_test_{1-train_ratio}.extxyz", test_db)

print("✅ Split saved: dataset_train.extxyz, dataset_test.extxyz")


✅ Total frames: 819
✅ Train frames: 409
✅ Test frames:  410
✅ Split saved: dataset_train.extxyz, dataset_test.extxyz


In [1]:
from ase import Atoms
from ase.io import read, write
from xtb.ase.calculator import XTB
from tqdm import tqdm
import numpy as np

# 1. Load merged db
db = read("./00_makedataset_merged_db.xyz", ":")

# 2. Add isolated atoms for EMIM + TFSI elements
isolated_atoms = ["H", "C", "N", "O", "F", "S"]
iso_db = [Atoms(sym) for sym in isolated_atoms]
for at in iso_db:
    at.info["config_type"] = "isolated_atom"

# 3. Run xTB for isolated atoms only (single point)
xtb_calc = XTB(method="GFN2-xTB")
for at in tqdm(iso_db, desc="Running xTB for isolated atoms"):
    at.calc = xtb_calc
    at.info["energy_xtb"] = at.get_potential_energy()
    at.arrays["forces_xtb"] = at.get_forces()

# 4. Merge isolated atoms + dataset
full_db = iso_db + db
print(f"✅ Total DB with isolated atoms: {len(full_db)} frames")

# 5. Train/Test Split (isolated atoms → always in train)
train_ratio = 0.5

iso_n = len(iso_db)                     # isolate atom count
rest_db = full_db[iso_n:]              # rest of real configs
np.random.seed(42)
perm = np.random.permutation(len(rest_db))
split_point = int(len(rest_db) * train_ratio)

train_db = iso_db + [rest_db[i] for i in perm[:split_point]]  # isolated atoms must be in train
test_db  = [rest_db[i] for i in perm[split_point:]]

print(f"✅ Train size = {len(train_db)} (including {iso_n} isolated atoms)")
print(f"✅ Test size  = {len(test_db)}")

# 6. Save datasets
write("dataset_train_0.5_.extxyz", train_db)
write("dataset_test_0.5_.extxyz", test_db)
print("✅ Saved: dataset_train_0.5.extxyz / dataset_test_0.5.extxyz")


Running xTB for isolated atoms: 100%|██████████| 6/6 [00:00<00:00, 17.80it/s]


✅ Total DB with isolated atoms: 819 frames
✅ Train size = 412 (including 6 isolated atoms)
✅ Test size  = 407
✅ Saved: dataset_train_0.5.extxyz / dataset_test_0.5.extxyz


In [2]:
from ase.io import read

db = read("dataset_train_0.5_.extxyz", ":")

# extract isolated atom energies
E0 = {}
for at in db:
    if at.info.get("config_type") == "isolated_atom":
        symbol = at.get_chemical_symbols()[0]
        E0[symbol] = at.info["energy_xtb"]

print("✅ E0 atomic reference energies:")
for k, v in E0.items():
    print(f"  {k}: {v}")


✅ E0 atomic reference energies:
  H: -10.707211383396714
  C: -48.847445262804705
  N: -71.00681805517411
  O: -102.57117256025786
  F: -125.69864294466228
  S: -85.66881795502964
